# 选座请求离线验证
先核对签名与表单序列化。本 notebook 不发送 HTTP 请求。submit_enc 通过隐藏输入提供，不写入代码。

## URL

In [ ]:
home = "https://office.chaoxing.com/front/third/apps/seat/index?fidEnc=35bbd135397006a8&mappId=4109435"
# 使用自己的登录会话访问，不保存历史 token。


In [ ]:
import hashlib
from getpass import getpass
from urllib.parse import urlencode

params = {
    "deptIdEnc": "35bbd135397006a8",
    "roomId": "6299",
    "day": "2026-09-16",
    "startTime": "17:30",
    "endTime": "18:00",
    "seatNum": "196",
    "captcha": "",
    "wyToken": "",
}

def verify_param(params: dict[str, str], submit_enc: str) -> str:
    if not all(isinstance(v, str) for v in params.values()):
        raise TypeError("此版本要求参数值均为字符串")
    text = "".join(f"[{key}={params[key]}]" for key in sorted(params))
    text += f"[{submit_enc}]"
    return hashlib.md5(text.encode("utf-8")).hexdigest()

In [ ]:
submit_enc = getpass("粘贴同一次页面中的完整 submit_enc：")
assert submit_enc, "submit_enc 不能为空"
enc = verify_param(params, submit_enc)
print("签名已在内存生成；不输出凭证")
assert verify_param(dict(reversed(list(params.items()))), submit_enc) == enc
assert verify_param({**params, "seatNum": "197"}, submit_enc) != enc
print("字段重排和单字段变化检查通过")

In [ ]:
body = urlencode({**params, "enc": enc})
print("表单已在内存生成；长度:", len(body))
# 只显示请求体，不发送。历史样本不代表当前可预约状态。